# Practical 4 – MovieLens 25M, version économe en RAM

**À faire d'abord : `Runtime → Restart session`** (les fichiers train/test sont déjà sur Drive, rien n'est perdu).
Ce notebook est autonome : il n'a pas besoin des anciennes cellules.

Pourquoi c'est léger : les listes Python de tuples `(i, r)` coûtent ~100 octets par note (≈ 5 Go pour 25 M × 2 listes).
Ici les notes sont stockées en matrices creuses CSR (`int32` + `float32`, 8 octets par note, ≈ 360 Mo au total),
et l'ALS est écrit en numba (parallèle, sans tableaux temporaires).

In [ ]:
import gc, os, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from numba import njit, prange
from google.colab import drive
drive.mount('/content/drive')

FOLDER      = '/content/drive/MyDrive/ml-25m'
movies_file = f'{FOLDER}/movies.csv'
train_file  = f'{FOLDER}/4train_ratings_90.csv'   # split 90/10 par utilisateur (déjà fait)
test_file   = f'{FOLDER}/4test_ratings_10.csv'

## 1. Split train / test
Le split existant (`4train_ratings_90.csv` / `4test_ratings_10.csv`) est déjà sensé : **90 % / 10 % des notes de chaque utilisateur**, tirées au hasard.
Chaque utilisateur (≥ 20 notes) a donc des notes dans le train (pour apprendre son vecteur) **et** dans le test (pour mesurer l'erreur).
Un split aléatoire global pourrait laisser des utilisateurs sans aucune note en train. On le recharge simplement, en format compact.

In [ ]:
movies   = pd.read_csv(movies_file, usecols=["movieId", "title"])
item_ids = movies.movieId.values.astype(np.int64)
titles   = movies.title.values
n_item   = len(item_ids)
item_map = np.full(item_ids.max() + 1, -1, dtype=np.int32)      # movieId -> indice
item_map[item_ids] = np.arange(n_item, dtype=np.int32)
N_USER   = 162541                                               # userId 1..162541, contigus

def load_csr(path, chunksize=4_000_000):
    us, its, rs = [], [], []
    for ch in pd.read_csv(path, usecols=["userId", "movieId", "rating"],
                          dtype={"userId": np.int32, "movieId": np.int32, "rating": np.float32},
                          chunksize=chunksize):
        us.append(ch.userId.values - 1)
        its.append(item_map[ch.movieId.values])
        rs.append(ch.rating.values)
    u, i, r = np.concatenate(us), np.concatenate(its), np.concatenate(rs)
    del us, its, rs
    assert u.max() < N_USER and i.min() >= 0
    return csr_matrix((r, (u, i)), shape=(N_USER, n_item))

Ru  = load_csr(train_file)            # lignes = utilisateurs
Ri  = Ru.T.tocsr()                    # lignes = films
Rte = load_csr(test_file)
gc.collect()

mu = float(Ru.data.mean(dtype=np.float64))                      # moyenne calculée sur le TRAIN seulement
item_cnt_train = np.diff(Ri.indptr)                             # nb de notes train par film
print(f"train {Ru.nnz:,}  test {Rte.nnz:,}  mu {mu:.4f}")
print(f"mémoire CSR : {(Ru.data.nbytes+Ru.indices.nbytes+Ri.data.nbytes+Ri.indices.nbytes+Rte.data.nbytes+Rte.indices.nbytes)/1e6:.0f} Mo")

## 2. ALS parallèle (numba)
Même modèle et mêmes mises à jour que ta Practical 3 : `r ≈ μ + b_u + b_i + U_u · V_i`, avec les 4 mises à jour (biais utilisateurs, biais films, U, V).
J'ai vérifié que ces noyaux donnent **exactement** les mêmes résultats que ta boucle Python sur un petit jeu de données (écart ~1e-17).
Un seul noyau sert pour les deux côtés (utilisateurs/films), car le modèle est symétrique.

In [ ]:
@njit
def chol_solve(A, b, k):
    for i in range(k):
        for j in range(i + 1):
            s = A[i, j]
            for p in range(j):
                s -= A[i, p] * A[j, p]
            if i == j:
                A[i, i] = np.sqrt(s)
            else:
                A[i, j] = s / A[j, j]
    y = np.empty(k)
    for i in range(k):
        s = b[i]
        for p in range(i):
            s -= A[i, p] * y[p]
        y[i] = s / A[i, i]
    x = np.empty(k)
    for i in range(k - 1, -1, -1):
        s = y[i]
        for p in range(i + 1, k):
            s -= A[p, i] * x[p]
        x[i] = s / A[i, i]
    return x

@njit(parallel=True, fastmath=True)
def update_bias(indptr, indices, data, F_row, F_col, b_col, mu, lam, gam, b_row):
    k = F_row.shape[1]
    for m in prange(len(indptr) - 1):
        a = indptr[m]
        e = indptr[m + 1]
        if e > a:
            s = 0.0
            for p in range(a, e):
                j = indices[p]
                d = 0.0
                for f in range(k):
                    d += F_row[m, f] * F_col[j, f]
                s += data[p] - mu - b_col[j] - d
            b_row[m] = lam * s / (lam * (e - a) + gam)

@njit(parallel=True, fastmath=True)
def update_factors(indptr, indices, data, b_row, b_col, mu, F_col, lam, tau, F_row):
    k = F_row.shape[1]
    for m in prange(len(indptr) - 1):
        a = indptr[m]
        e = indptr[m + 1]
        if e > a:
            A = np.zeros((k, k))
            rhs = np.zeros(k)
            for x in range(k):
                A[x, x] = tau
            for p in range(a, e):
                j = indices[p]
                res = data[p] - mu - b_row[m] - b_col[j]
                for x in range(k):
                    w = lam * F_col[j, x]
                    rhs[x] += w * res
                    for y in range(x + 1):
                        A[x, y] += w * F_col[j, y]
            F_row[m, :] = chol_solve(A, rhs, k)

@njit(parallel=True, fastmath=True)
def sse_rows(indptr, indices, data, F_row, F_col, b_row, b_col, mu):
    k = F_row.shape[1]
    out = np.zeros(len(indptr) - 1)
    for m in prange(len(indptr) - 1):
        s = 0.0
        for p in range(indptr[m], indptr[m + 1]):
            j = indices[p]
            d = 0.0
            for f in range(k):
                d += F_row[m, f] * F_col[j, f]
            err = data[p] - mu - b_row[m] - b_col[j] - d
            s += err * err
        out[m] = s
    return out

@njit(parallel=True)
def row_std(indptr, data):
    out = np.zeros(len(indptr) - 1)
    for m in prange(len(indptr) - 1):
        c = indptr[m + 1] - indptr[m]
        if c > 1:
            s = 0.0
            s2 = 0.0
            for p in range(indptr[m], indptr[m + 1]):
                x = data[p]
                s += x
                s2 += x * x
            out[m] = np.sqrt(max(s2 / c - (s / c) ** 2, 0.0))
    return out

def als(Ru, Ri, Rte, k, lam, gam, tau, iters, mu, seed=0, verbose=True):
    n_user, n_item = Ru.shape
    rng = np.random.default_rng(seed)
    U = 0.1 * rng.standard_normal((n_user, k))
    V = 0.1 * rng.standard_normal((n_item, k))
    U[np.diff(Ru.indptr) == 0] = 0.0
    V[np.diff(Ri.indptr) == 0] = 0.0
    bu = np.zeros(n_user); bi = np.zeros(n_item)
    N = Ru.nnz
    N_te = Rte.nnz if Rte is not None else 0
    hist = {"train": [], "test": [], "loss": []}
    t0 = time.time()
    for it in range(iters):
        update_bias(Ru.indptr, Ru.indices, Ru.data, U, V, bi, mu, lam, gam, bu)
        update_bias(Ri.indptr, Ri.indices, Ri.data, V, U, bu, mu, lam, gam, bi)
        update_factors(Ru.indptr, Ru.indices, Ru.data, bu, bi, mu, V, lam, tau, U)
        update_factors(Ri.indptr, Ri.indices, Ri.data, bi, bu, mu, U, lam, tau, V)
        sse_tr = sse_rows(Ru.indptr, Ru.indices, Ru.data, U, V, bu, bi, mu)
        hist["train"].append(np.sqrt(sse_tr.sum() / N))
        nb, nf = n_user + n_item, n_user + n_item
        hist["loss"].append(
            lam / 2 * sse_tr.sum() - N / 2 * np.log(lam / (2 * np.pi))
            - nb / 2 * np.log(gam / (2 * np.pi)) + gam / 2 * (bu @ bu + bi @ bi)
            - nf * k / 2 * np.log(tau / (2 * np.pi)) + tau / 2 * ((U * U).sum() + (V * V).sum()))
        sse_te = None
        if Rte is not None:
            sse_te = sse_rows(Rte.indptr, Rte.indices, Rte.data, U, V, bu, bi, mu)
            hist["test"].append(np.sqrt(sse_te.sum() / N_te))
        if verbose:
            te = f"{hist['test'][-1]:.4f}" if Rte is not None else "-"
            print(f"  it {it+1:2d}  train {hist['train'][-1]:.4f}  test {te}  ({time.time()-t0:.0f}s)")
    return dict(U=U, V=V, bu=bu, bi=bi, mu=mu, k=k, lam=lam, gam=gam, tau=tau,
                hist=hist, sse_tr=sse_tr, sse_te=sse_te)

## 3. Overfitting : train vs test RMSE
Seul le **train** sert à optimiser ; le test n'est que mesuré (paramètres figés) à chaque itération.

Remarque : seuls les rapports τ/λ et γ/λ comptent (multiplier λ, τ, γ par la même constante ne change pas la solution MAP).
On fait donc varier **τ** avec λ=1. Le run `τ=10⁴` rend les facteurs ≈ 0 : c'est le **modèle à biais seuls**, notre référence « sous-apprentissage ».

In [ ]:
ITERS, LAM, GAM = 15, 1.0, 0.1
grid = [(10, 1.), (20, 1.), (10, 10.), (20, 10.), (10, 50.), (20, 50.), (10, 1e4)]   # (K, tau)
runs = {}
for k, tau in grid:
    print(f"== K={k}  tau={tau:g}")
    runs[(k, tau)] = als(Ru, Ri, Rte, k, LAM, GAM, tau, ITERS, mu, verbose=(tau != 1e4))
    gc.collect()

print(f"\n{'config':>14} {'train':>8} {'test':>8} {'écart':>8} {'meilleure it. test':>20}")
for (k, tau), r in runs.items():
    h = r['hist']
    print(f"K={k:<3} tau={tau:<6g} {h['train'][-1]:8.4f} {h['test'][-1]:8.4f} {h['test'][-1]-h['train'][-1]:8.4f}"
          f" {int(np.argmin(h['test']))+1:>12d} ({min(h['test']):.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
ref = runs[(10, 1e4)]['hist']
for ax, K in zip(axes, (10, 20)):
    for (k, tau), c in zip([g for g in grid if g[0] == K], plt.cm.tab10.colors):
        h = runs[(k, tau)]['hist']
        ax.plot(range(1, ITERS+1), h['train'], '-',  color=c, label=f'train τ={tau:g}')
        ax.plot(range(1, ITERS+1), h['test'],  '--', color=c, label=f'test τ={tau:g}')
    ax.plot(range(1, ITERS+1), ref['test'], ':', color='k', label='test, biais seuls')
    ax.set_title(f'K={K}'); ax.set_xlabel('itération'); ax.grid(alpha=.3); ax.legend(fontsize=8)
axes[0].set_ylabel('RMSE')
plt.tight_layout(); plt.show()

## 4. K=10 vs K=20 : pour quels utilisateurs ? Y a-t-il du sous-apprentissage ?
On regroupe les utilisateurs par nombre total de notes et on calcule le RMSE train / test par groupe (tout vient des erreurs par utilisateur déjà calculées, aucun recalcul lourd).

In [ ]:
edges  = [50, 100, 200, 500, 1000]
labels = ["20-49", "50-99", "100-199", "200-499", "500-999", "1000+"]
n_tr, n_te = np.diff(Ru.indptr), np.diff(Rte.indptr)
bins = np.digitize(n_tr + n_te, edges)
nb   = len(labels)
n_users_bin = np.bincount(bins, minlength=nb)

def rmse_by_bin(run):
    tr = np.sqrt(np.bincount(bins, run['sse_tr'], nb) / np.bincount(bins, n_tr, nb))
    te = np.sqrt(np.bincount(bins, run['sse_te'], nb) / np.bincount(bins, n_te, nb))
    return tr, te

show = [(10, 10.), (20, 10.), (10, 1.), (20, 1.), (10, 1e4)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(nb)
for key, c in zip(show, plt.cm.tab10.colors):
    tr, te = rmse_by_bin(runs[key])
    name = "biais seuls" if key[1] == 1e4 else f"K={key[0]}, τ={key[1]:g}"
    axes[0].plot(x, tr, '-o', color=c, label=name + ' train'); axes[0].plot(x, te, '--s', color=c, label=name + ' test')
    axes[1].plot(x, te - tr, '-o', color=c, label=name)
for ax, t in zip(axes, ("RMSE par groupe d'utilisateurs", "écart test − train (overfitting)")):
    ax.set_xticks(x); ax.set_xticklabels([f"{l}\n(n={n:,})" for l, n in zip(labels, n_users_bin)], fontsize=8)
    ax.set_title(t); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

print(f"{'groupe':>9} | " + " | ".join(f"{('K=%d τ=%g' % k) if k[1] != 1e4 else 'biais':>16}" for k in show))
tabs = {k: rmse_by_bin(runs[k]) for k in show}
for b in range(nb):
    print(f"{labels[b]:>9} | " + " | ".join(f"{tabs[k][0][b]:.3f} / {tabs[k][1][b]:.3f}  " for k in show))
print("(format : train / test)")

## 5. Entraînement sur les 25M notes
On choisit (K, τ) avec le **meilleur RMSE test** (on ignore le run « biais seuls »), puis on réentraîne sur train + test.
Le nombre d'itérations est celui qui minimisait le test. Le modèle est sauvegardé sur Drive (quelques dizaines de Mo).

In [ ]:
best_key = min((k for k in runs if k[1] < 1e3), key=lambda k: min(runs[k]['hist']['test']))
n_it = int(np.argmin(runs[best_key]['hist']['test'])) + 1
print("meilleure config :", best_key, "itérations :", n_it)

Ru_full = (Ru + Rte).tocsr()
del Ru, Ri, Rte; gc.collect()
Ri_full = Ru_full.T.tocsr()
mu_full = float(Ru_full.data.mean(dtype=np.float64))
item_cnt = np.diff(Ri_full.indptr)

model = als(Ru_full, Ri_full, None, best_key[0], LAM, GAM, best_key[1], n_it, mu_full)
np.savez(f'{FOLDER}/model_full.npz', U=model['U'], V=model['V'], bu=model['bu'], bi=model['bi'], mu=mu_full)

## 6. Utilisateur fictif : les recommandations ont-elles du sens ?
Pour un utilisateur qui n'est pas dans le modèle, on ne ré-entraîne rien : on **résout son vecteur de traits** avec V et les biais films fixés
(c'est exactement la mise à jour de U_m, sur ses seules notes). Score : `U·V_n + w·b_n`, avec w = 1 puis 0.05, avec ou sans filtre sur le nombre de notes.

In [ ]:
def find(sub, top=8):
    m = np.where(pd.Series(titles).str.contains(sub, case=False, regex=False).values)[0]
    m = m[np.argsort(-item_cnt[m])][:top]
    return [(int(j), titles[j], int(item_cnt[j])) for j in m]

def fold_in(liked, model, n_inner=10, use_bias=False):
    """liked = {indice_film: note}. Retourne le vecteur de traits (et le biais) de l'utilisateur fictif."""
    V, bi, mu_, lam, gam, tau = model['V'], model['bi'], model['mu'], model['lam'], model['gam'], model['tau']
    idx = np.array(list(liked.keys())); r = np.array(list(liked.values()), dtype=float)
    Vs = V[idx]; A = tau * np.eye(V.shape[1]) + lam * Vs.T @ Vs; bu_ = 0.0
    for _ in range(n_inner):
        u = np.linalg.solve(A, lam * Vs.T @ (r - mu_ - bu_ - bi[idx]))
        if use_bias:
            bu_ = lam * np.sum(r - mu_ - bi[idx] - Vs @ u) / (lam * len(r) + gam)
    return u, bu_

def recommend(u, model, w_bias=0.05, min_ratings=0, exclude=(), top=10):
    score = model['V'] @ u + w_bias * model['bi']
    score[item_cnt < min_ratings] = -np.inf
    score[list(exclude)] = -np.inf
    top = min(top, int(np.isfinite(score).sum()))
    sel = np.argpartition(-score, top)[:top]; sel = sel[np.argsort(-score[sel])]
    return [(titles[j], int(item_cnt[j]), round(float(score[j]), 3)) for j in sel]

model['mu'] = mu_full
for s in ["Fellowship of the Ring", "Two Towers", "Return of the King"]:
    print(find(s, 3))

In [ ]:
# Change ici : un seul film à 5 étoiles (ou plusieurs)
fellowship = find("Fellowship of the Ring", 1)[0][0]
dummy = {fellowship: 5.0}

u_dummy, _ = fold_in(dummy, model)
for w, mr in [(1.0, 0), (0.05, 0), (0.05, 100), (0.0, 100)]:
    print(f"\n--- poids du biais film = {w}, filtre >= {mr} notes ---")
    for t, c, s in recommend(u_dummy, model, w, mr, exclude=dummy.keys()):
        print(f"  {s:7.3f}  {t}  ({c} notes)")

## 7. Films polarisants
Un film dont le vecteur de traits `v` est **long** sépare fortement les utilisateurs (selon les goûts, le produit `U·v` est très positif ou très négatif) :
savoir si on l'aime ou non en dit beaucoup sur nous. On compare la norme ‖v‖, l'écart-type des notes et le nombre de notes.

In [ ]:
norms = np.linalg.norm(model['V'], axis=1)
stds  = row_std(Ri_full.indptr, Ri_full.data)

def show_top(mask, n=15, title=""):
    cand = np.where(mask)[0]; cand = cand[np.argsort(-norms[cand])][:n]
    print(f"\n{title}\n{'‖v‖':>6} {'std':>5} {'notes':>7}  titre")
    for j in cand:
        print(f"{norms[j]:6.2f} {stds[j]:5.2f} {item_cnt[j]:7d}  {titles[j]}")

show_top(item_cnt >= 1,    title="Sans filtre")
show_top(item_cnt >= 1000, title="Films avec >= 1000 notes")
print("\nnorme moyenne des films :", norms[item_cnt >= 1000].mean().round(3))